# Scikit learn gaussian process regression for Air Quality Index

a notebook to optmize a gaussian process using Scikit learns implementation, for comparison againt MuyGPs

In [1]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, ConstantKernel, WhiteKernel
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split, KFold
import time

## Import/Setup data

In [2]:
BASE_DIR = os.path.abspath("..") + "\\"
aqi_data = np.genfromtxt(BASE_DIR + "data\\biggest_day_aqi_coords_scaled_noAH.csv", delimiter=',', skip_header=1)
#print(aqi_data[:10])

#randomly split the data into features and responces,
aqi_x, aqi_y = aqi_data[:, :-1], aqi_data[:, -1]

In [3]:
#setup metrics storage, to keep track of performance
optimization_times = []
rmse_scores = []
mean_variances = []
mean_conf_intervals = []

kfold = KFold(n_splits=10, shuffle=True, random_state=24)

In [4]:
#cycle through all folds
for fold, (train_idx, test_idx) in enumerate(kfold.split(aqi_x)):
    #seperate into training testing feature and value sets.
    aqi_x_train, aqi_x_test = aqi_x[train_idx], aqi_x[test_idx]
    aqi_y_train, aqi_y_test = aqi_y[train_idx], aqi_y[test_idx]
    #use the mean of the training points to provide basic de-meaning of the training points.
    aqi_y_mean = aqi_y_train.mean()
    aqi_y_train = aqi_y_train - aqi_y_mean

    aqi_var = float(aqi_y_train.var())

    #create a kernel
    kernel = (
        ConstantKernel(aqi_var, constant_value_bounds=(aqi_var/10, aqi_var * 10))
        * Matern(length_scale=0.7, length_scale_bounds=(0.01, 0.20), nu=1.5)
        + WhiteKernel(noise_level=aqi_var * 0.05,
                      noise_level_bounds=(aqi_var * 0.003, aqi_var * 0.25))
    )

    #Define the model
    scikit_gp = GaussianProcessRegressor(kernel=kernel, normalize_y=False, n_restarts_optimizer=10, random_state=24)

    #Start training
    optimize_start = time.time()
    scikit_gp.fit(aqi_x_train, aqi_y_train)
    optimize_end = time.time()
    optimize_time = optimize_end - optimize_start
    print(f"Gaussian Process optimization took {optimize_time} seconds")

    #predict values
    scikit_predicted_centered, scikit_sigma = scikit_gp.predict(aqi_x_test, return_std=True)
    scikit_predicted = scikit_predicted_centered + aqi_y_mean

    #calculate metrics
    # calculate values that equate to the ones used to rank muyGPs. Scikit returns the variances already square rooted.
    confidence_intervals = scikit_sigma * 1.96
    mean_confidence_interval = confidence_intervals.mean()
    # Evaluate against the ORIGINAL (un-centered) test responses.
    coverage = np.count_nonzero(
        np.abs(aqi_y_test - scikit_predicted) < confidence_intervals
    ) / aqi_y_test.size
    rmse = np.sqrt(np.mean((aqi_y_test - scikit_predicted) ** 2))

    rmse_scores.append(rmse)
    mean_variances.append(np.mean(scikit_sigma))
    mean_conf_intervals.append(np.mean(confidence_intervals)) 
    optimization_times.append(optimize_time)
    
#print out the results
print(f"Avg RMSE: {np.mean(rmse_scores):.4f}")
print(f"Avg Variances: {np.mean(mean_variances):.4f}")
print(f"Avg Confidence Intervals: {np.mean(mean_conf_intervals):.4f}")
print(f"Avg Optimization Time: {np.mean(optimization_times):.4f}")

I:\School\CSC433_AdvancedAI\AQI_MuyGPs\GP_env\Lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified upper bound 67.15434126878964. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Gaussian Process optimization took 24.439332246780396 seconds


I:\School\CSC433_AdvancedAI\AQI_MuyGPs\GP_env\Lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified upper bound 68.5263297375634. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Gaussian Process optimization took 22.684368133544922 seconds


I:\School\CSC433_AdvancedAI\AQI_MuyGPs\GP_env\Lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified upper bound 67.64594690753005. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


...

I:\School\CSC433_AdvancedAI\AQI_MuyGPs\GP_env\Lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified upper bound 66.30764804510659. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Gaussian Process optimization took 23.97344732284546 seconds


I:\School\CSC433_AdvancedAI\AQI_MuyGPs\GP_env\Lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified upper bound 67.82799744897959. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Gaussian Process optimization took 22.797230005264282 seconds
Gaussian Process optimization took 22.57762122154236 seconds
Avg RMSE: 14.5487
Avg Variances: 12.9124
Avg Confidence Intervals: 25.3082
Avg Optimization Time: 23.0998


I:\School\CSC433_AdvancedAI\AQI_MuyGPs\GP_env\Lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified upper bound 67.21220611203665. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


## Selecting a kernel

## Creating the GP model

## Training the model

## Making Predictions

In [5]:
# calculate values that equate to the ones used to rank muyGPs. Scikit returns the variances already square rooted.
confidence_intervals = scikit_sigma * 1.96
mean_confidence_interval = confidence_intervals.mean()

# Evaluate against the ORIGINAL (un-centered) test responses.
coverage = np.count_nonzero(
    np.abs(aqi_y_test - scikit_predicted) < confidence_intervals
) / aqi_y_test.size

rmse = np.sqrt(np.mean((aqi_y_test - scikit_predicted) ** 2))
print(f"RMSE: {rmse:.4f}")
print(f"Coverage: {coverage:.4f}")
print(f"Mean Confidence Interval: {mean_confidence_interval:.4f}")

RMSE: 14.5146
Coverage: 0.9352
Mean Confidence Interval: 25.3412
